Install Packages and Import Libraries

In [ ]:
# Install required packages for Azure OpenAI and Pydantic structured outputs.
%pip install -r ../requirements.txt

In [ ]:
# Import libraries for prompt generation and Azure OpenAI integration.
import os
import json
from pathlib import Path
from openai import AzureOpenAI
from pydantic import BaseModel, Field, ConfigDict
from typing import List
from dotenv import load_dotenv

Define Structured Output Schema

In [ ]:
'''
Utilized Pydantic schemas for Azure OpenAI structured outputs to enforce response structure.
See: https://learn.microsoft.com/en-us/azure/ai-services/openai/how-to/structured-outputs
'''

class PromptEnsemble(BaseModel):
    prompts: List[str] = Field(..., description='Textual descriptions of visual discriminative features.')
    model_config = ConfigDict(extra='forbid')

Load Azure OpenAI Credentials

In [ ]:
# Load Azure OpenAI credentials from environment file.
project_root = Path('../').resolve()
env_path = project_root / '.env'
load_dotenv(env_path, override=True)

azure_openai_endpoint = os.getenv('azure_openai_endpoint')
azure_openai_api_key = os.getenv('azure_openai_api_key')
azure_openai_api_version = '2024-12-01-preview'
azure_openai_deployment = 'gpt-5.4'

client = AzureOpenAI(
    azure_endpoint=azure_openai_endpoint,
    api_key=azure_openai_api_key,
    api_version=azure_openai_api_version,
)

Generate Class Specific Prompt Ensembles

In [ ]:
'''
Referred to BiomedCoOp for LLM-based prompt ensembling using GPT-5.
See: https://github.com/HealthX-Lab/BiomedCoOp/
See: https://arxiv.org/pdf/2411.15232

Note: I followed their approach while enforcing consistent template structure. 

We also use custom BI-RADS ultrasound lexicon descriptors for the prompt context to cover shape, orientation, margin, echo pattern, and posterior features. 
See: https://radiologyassistant.nl/breast/bi-rads/bi-rads-for-mammography-and-ultrasound-2013

As a group we may need to utilize BI-RADS context in the generation phase rather than template generation followed by manual editing.
See: https://www.sciencedirect.com/science/article/pii/S0925231226011446
'''

def generate_prompt_ensemble(class_name, modality, num_prompts=10):
    # Map class names to full labels used in prompts
    class_labels = {
        'benign tumor': 'benign breast tumor',
        'malignant tumor': 'malignant breast tumor',
        'normal scan': 'normal breast ultrasound scan'
    }
    
    full_label = class_labels.get(class_name, class_name)
    
    user_prompt = f'''
Generate {num_prompts} textual descriptions of visual discriminative features for distinct medical cases of {class_name} found in {modality}.

Example format - "an ultrasound image showing well-defined, smooth margins, indicating a {full_label}."
'''

    response = client.beta.chat.completions.parse(
        model=azure_openai_deployment,
        messages=[{'role': 'user', 'content': user_prompt}],
        response_format=PromptEnsemble,
    )

    parsed = response.choices[0].message.parsed
    prompts = parsed.prompts

    if len(prompts) != num_prompts:
        raise ValueError(f'expected {num_prompts} prompts for {class_name}, got {len(prompts)}')

    return prompts

Generate Prompt Ensembles for Each Target Class

In [ ]:
# Generate 10 prompts per class for zero-shot CLIP ensemble evaluation (average embeddings).
modality = 'breast ultrasound'
num_prompts_per_class = 10
classes = ['benign tumor', 'malignant tumor', 'normal scan']

prompt_registry = {}

for class_name in classes:
    print(f'generating {num_prompts_per_class} prompts for {class_name}')

    prompts = generate_prompt_ensemble(
        class_name=class_name,
        modality=modality,
        num_prompts=num_prompts_per_class,
    )

    prompt_registry[class_name] = prompts

    print(f'{class_name} - {len(prompts)} prompts generated')

print(f'\ntotal prompts generated - {sum(len(p) for p in prompt_registry.values())}')

Save the Prompt Registry

In [ ]:
# Save prompt registry to Python file for import in training pipeline.
registry_path = project_root / 'src' / 'local' / 'prompting' / 'prompt_registry.py'
registry_path.parent.mkdir(parents=True, exist_ok=True)

with open(registry_path, 'w') as f:
    f.write('PROMPT_REGISTRY = ')
    f.write(json.dumps(prompt_registry, indent=2))
    f.write('\n')

print(f'prompt registry saved to - {registry_path}')  

print(f'\nregistry structure:')
for class_name, prompts in prompt_registry.items():
    print(f'* {class_name} - {len(prompts)} prompts')